In [1]:
# ============================================================
# Cell 1 — Imports
# ============================================================

import torch
import torch.nn as nn

from course_tasks.hw2.ppo import PPO

# Adjust this import only if GaussianActor lives somewhere else
try:
    from course_tasks.hw2.reinforce import GaussianActor
except ImportError:
    from course_tasks.hw2.ppo import GaussianActor

torch.manual_seed(42)

print("PyTorch:", torch.__version__)
print("Imports OK")

/u/kchan8/cs498/course/.venv/lib/python3.13/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


PyTorch: 2.14.0+cu130
Imports OK


In [2]:
# ============================================================
# Cell 2 — Tiny fake RolloutStorage
# ============================================================
#
# We don't need the full environment for testing returns / GAE.
# We only need the fields that compute_returns() and
# compute_advantages() access.
# ============================================================

class TinyStorage:
    def __init__(self, T, N):
        self.num_transitions_per_env = T

        self.rewards = torch.zeros(T, N)
        self.dones = torch.zeros(T, N)
        self.values = torch.zeros(T, N)
        self.returns = torch.zeros(T, N)
        self.advantages = torch.zeros(T, N)


class ZeroCritic(nn.Module):
    """Always predicts V(s) = 0."""
    def forward(self, obs):
        return torch.zeros(obs.shape[0], device=obs.device)


class ConstantCritic(nn.Module):
    """Always predicts the same value."""
    def __init__(self, value):
        super().__init__()
        self.value = value

    def forward(self, obs):
        return torch.full(
            (obs.shape[0],),
            self.value,
            device=obs.device,
        )


def make_test_ppo(storage, critic, gamma=0.9, lam=1.0):
    """
    Make a minimal PPO object without calling PPO.__init__().
    This lets us directly test YOUR compute_returns() and
    compute_advantages().
    """
    alg = PPO.__new__(PPO)

    alg.storage = storage
    alg.critic = critic
    alg.gamma = gamma
    alg.lam = lam
    alg.normalize_advantage = False

    return alg


print("Testing helpers ready")

Testing helpers ready


In [3]:
# ============================================================
# Cell 3 — Test 1
# lambda = 1, V = 0 => GAE == discounted return-to-go
# ============================================================

T = 3
N = 1
OBS_DIM = 5

st = TinyStorage(T, N)

st.rewards[:, 0] = torch.tensor([
    1.0,
    2.0,
    3.0,
])

# Final step is a true terminal
st.dones[:, 0] = torch.tensor([
    0.0,
    0.0,
    1.0,
])

# V(s) = 0 everywhere
st.values[:, 0] = 0.0

alg = make_test_ppo(
    storage=st,
    critic=ZeroCritic(),
    gamma=0.9,
    lam=1.0,
)

last_obs = torch.zeros(N, OBS_DIM)

alg.compute_returns(last_obs)
alg.compute_advantages()

expected = torch.tensor([
    5.23,
    4.70,
    3.00,
])

print("returns:   ", st.returns[:, 0])
print("advantages:", st.advantages[:, 0])
print("expected:  ", expected)

assert torch.allclose(
    st.returns[:, 0],
    expected,
    atol=1e-5,
)

assert torch.allclose(
    st.advantages[:, 0],
    expected,
    atol=1e-5,
)

print("PASS: lambda=1, V=0 reproduces return-to-go")

returns:    tensor([5.2300, 4.7000, 3.0000])
advantages: tensor([5.2300, 4.7000, 3.0000])
expected:   tensor([5.2300, 4.7000, 3.0000])
PASS: lambda=1, V=0 reproduces return-to-go


In [4]:
# ============================================================
# Cell 4 — Test 2
# done in middle of buffer stops credit leakage
# ============================================================

T = 4
N = 1

st = TinyStorage(T, N)

st.rewards[:, 0] = torch.tensor([
    1.0,
    2.0,
    10.0,
    20.0,
])

st.dones[:, 0] = torch.tensor([
    0.0,
    1.0,    # episode 1 ends here
    0.0,
    1.0,    # episode 2 ends here
])

st.values[:, 0] = 0.0

alg = make_test_ppo(
    storage=st,
    critic=ZeroCritic(),
    gamma=0.9,
    lam=1.0,
)

last_obs = torch.zeros(N, OBS_DIM)

alg.compute_returns(last_obs)
alg.compute_advantages()

# Episode 1:
# G1 = 2
# G0 = 1 + .9*2 = 2.8
#
# Episode 2:
# G3 = 20
# G2 = 10 + .9*20 = 28

expected = torch.tensor([
    2.8,
    2.0,
    28.0,
    20.0,
])

print("returns:   ", st.returns[:, 0])
print("advantages:", st.advantages[:, 0])
print("expected:  ", expected)

assert torch.allclose(
    st.returns[:, 0],
    expected,
    atol=1e-5,
)

assert torch.allclose(
    st.advantages[:, 0],
    expected,
    atol=1e-5,
)

print("PASS: done stops reward/advantage leakage")

returns:    tensor([ 2.8000,  2.0000, 28.0000, 20.0000])
advantages: tensor([ 2.8000,  2.0000, 28.0000, 20.0000])
expected:   tensor([ 2.8000,  2.0000, 28.0000, 20.0000])
PASS: done stops reward/advantage leakage


In [5]:
# ============================================================
# Cell 5 — Test 3A
# Mid-episode rollout MUST use V(last_obs)
# ============================================================

T = 2
N = 1

st = TinyStorage(T, N)

st.rewards[:, 0] = torch.tensor([
    1.0,
    1.0,
])

# Neither step is terminal.
st.dones[:, 0] = torch.tensor([
    0.0,
    0.0,
])

alg = make_test_ppo(
    storage=st,
    critic=ConstantCritic(5.0),
    gamma=0.9,
    lam=1.0,
)

last_obs = torch.zeros(N, OBS_DIM)

alg.compute_returns(last_obs)

expected = torch.tensor([
    5.95,
    5.50,
])

print("returns: ", st.returns[:, 0])
print("expected:", expected)
print("last_value:", alg.last_value)

assert torch.allclose(
    st.returns[:, 0],
    expected,
    atol=1e-5,
)

print("PASS: mid-episode rollout bootstraps from last_obs")

returns:  tensor([5.9500, 5.5000])
expected: tensor([5.9500, 5.5000])
last_value: tensor([5.])
PASS: mid-episode rollout bootstraps from last_obs


In [6]:
# ============================================================
# Cell 6 — Test 3B
# Terminal final step MUST NOT use V(last_obs)
# ============================================================

T = 2
N = 1

st = TinyStorage(T, N)

st.rewards[:, 0] = torch.tensor([
    1.0,
    1.0,
])

st.dones[:, 0] = torch.tensor([
    0.0,
    1.0,       # true terminal
])

alg = make_test_ppo(
    storage=st,
    critic=ConstantCritic(5.0),
    gamma=0.9,
    lam=1.0,
)

last_obs = torch.zeros(N, OBS_DIM)

alg.compute_returns(last_obs)

# Final bootstrap is killed by (1 - done)
#
# G1 = 1
# G0 = 1 + .9*1 = 1.9

expected = torch.tensor([
    1.9,
    1.0,
])

print("returns: ", st.returns[:, 0])
print("expected:", expected)

assert torch.allclose(
    st.returns[:, 0],
    expected,
    atol=1e-5,
)

print("PASS: terminal step ignores last_obs value")

returns:  tensor([1.9000, 1.0000])
expected: tensor([1.9000, 1.0000])
PASS: terminal step ignores last_obs value


In [7]:
# ============================================================
# Cell 7 — Test 4
# Initial PPO ratio should be 1
# ============================================================

OBS_DIM = 5
ACTION_DIM = 1
BATCH_SIZE = 64

actor = GaussianActor(
    obs_dim=OBS_DIM,
    action_dim=ACTION_DIM,
)

obs = torch.randn(BATCH_SIZE, OBS_DIM)

# This represents rollout collection.
with torch.inference_mode():
    actions, old_log_probs = actor.act(obs)

# Re-score the SAME actions under the SAME current policy.
log_probs, entropy = actor.evaluate_actions(
    obs,
    actions,
)

ratio = torch.exp(
    log_probs - old_log_probs
)

print("ratio mean:         ", ratio.mean().item())
print("ratio min:          ", ratio.min().item())
print("ratio max:          ", ratio.max().item())
print(
    "max |ratio - 1|:   ",
    (ratio - 1.0).abs().max().item(),
)

assert torch.allclose(
    ratio,
    torch.ones_like(ratio),
    atol=1e-6,
)

print("PASS: initial PPO ratio is 1")

ratio mean:          1.0
ratio min:           1.0
ratio max:           1.0
max |ratio - 1|:    0.0
PASS: initial PPO ratio is 1


In [8]:
# ============================================================
# Cell 8 — Test 5
# PPO gradient == REINFORCE gradient at initial policy
# ============================================================

torch.manual_seed(42)

actor = GaussianActor(
    obs_dim=OBS_DIM,
    action_dim=ACTION_DIM,
)

obs = torch.randn(BATCH_SIZE, OBS_DIM)

with torch.inference_mode():
    actions, old_log_probs = actor.act(obs)

# Fixed advantages for the test.
advantages = torch.linspace(
    -1.0,
    1.0,
    BATCH_SIZE,
)

params = [
    p for p in actor.parameters()
    if p.requires_grad
]


# ------------------------------------------------------------
# REINFORCE gradient
# ------------------------------------------------------------

log_probs_reinforce, _ = actor.evaluate_actions(
    obs,
    actions,
)

reinforce_loss = -(
    log_probs_reinforce * advantages
).mean()

reinforce_grads = torch.autograd.grad(
    reinforce_loss,
    params,
)


# ------------------------------------------------------------
# PPO gradient
# ------------------------------------------------------------

# Fresh forward pass because autograd graph from above was consumed.
log_probs_ppo, _ = actor.evaluate_actions(
    obs,
    actions,
)

ratio = torch.exp(
    log_probs_ppo - old_log_probs
)

# Wide clip so it can never matter here.
clip_param = 100.0

unclipped = ratio * advantages

clipped = (
    torch.clamp(
        ratio,
        1.0 - clip_param,
        1.0 + clip_param,
    )
    * advantages
)

ppo_loss = -torch.minimum(
    unclipped,
    clipped,
).mean()

ppo_grads = torch.autograd.grad(
    ppo_loss,
    params,
)


# ------------------------------------------------------------
# Compare gradients
# ------------------------------------------------------------

max_differences = []

for i, (g_reinforce, g_ppo) in enumerate(
    zip(reinforce_grads, ppo_grads)
):
    diff = (
        g_reinforce - g_ppo
    ).abs().max().item()

    max_differences.append(diff)

    print(
        f"parameter {i}: max gradient difference = {diff:.10f}"
    )

max_diff = max(max_differences)

print()
print("overall max difference:", max_diff)

assert max_diff < 1e-5

print("PASS: PPO and REINFORCE gradients match")

parameter 0: max gradient difference = 0.0000000000
parameter 1: max gradient difference = 0.0000000000
parameter 2: max gradient difference = 0.0000000000
parameter 3: max gradient difference = 0.0000000000
parameter 4: max gradient difference = 0.0000000000
parameter 5: max gradient difference = 0.0000000000
parameter 6: max gradient difference = 0.0000000000

overall max difference: 0.0
PASS: PPO and REINFORCE gradients match


In [9]:
# ============================================================
# Cell 9 — Test 6
# Ratio should move after an optimizer update
# ============================================================

torch.manual_seed(42)

actor = GaussianActor(
    obs_dim=OBS_DIM,
    action_dim=ACTION_DIM,
)

optimizer = torch.optim.Adam(
    actor.parameters(),
    lr=1e-3,
)

obs = torch.randn(BATCH_SIZE, OBS_DIM)

with torch.inference_mode():
    actions, old_log_probs = actor.act(obs)

advantages = torch.linspace(
    -1.0,
    1.0,
    BATCH_SIZE,
)

clip_param = 0.2


def ratio_diagnostics(actor, obs, actions, old_log_probs):
    with torch.no_grad():
        new_log_probs, _ = actor.evaluate_actions(
            obs,
            actions,
        )

        ratio = torch.exp(
            new_log_probs - old_log_probs
        )

        clip_fraction = (
            (ratio < 1.0 - clip_param)
            |
            (ratio > 1.0 + clip_param)
        ).float().mean()

        return {
            "mean_ratio": ratio.mean().item(),
            "mean_abs_drift": (
                ratio - 1.0
            ).abs().mean().item(),
            "max_abs_drift": (
                ratio - 1.0
            ).abs().max().item(),
            "clip_fraction": clip_fraction.item(),
        }


print("BEFORE update:")
print(
    ratio_diagnostics(
        actor,
        obs,
        actions,
        old_log_probs,
    )
)


# One PPO policy update
log_probs, entropy = actor.evaluate_actions(
    obs,
    actions,
)

ratio = torch.exp(
    log_probs - old_log_probs
)

unclipped = ratio * advantages

clipped = (
    torch.clamp(
        ratio,
        1.0 - clip_param,
        1.0 + clip_param,
    )
    * advantages
)

policy_loss = -torch.minimum(
    unclipped,
    clipped,
).mean()

optimizer.zero_grad()
policy_loss.backward()
optimizer.step()


print()
print("AFTER update:")
after = ratio_diagnostics(
    actor,
    obs,
    actions,
    old_log_probs,
)

print(after)

assert after["max_abs_drift"] > 0

print("PASS: policy changed and ratio moved away from 1")

BEFORE update:
{'mean_ratio': 1.0, 'mean_abs_drift': 0.0, 'max_abs_drift': 0.0, 'clip_fraction': 0.0}

AFTER update:
{'mean_ratio': 1.0006670951843262, 'mean_abs_drift': 0.03246069326996803, 'max_abs_drift': 0.21580791473388672, 'clip_fraction': 0.015625}
PASS: policy changed and ratio moved away from 1


In [10]:
# ============================================================
# Cell 10 — Diagnostic formulas for your real PPO.update()
# ============================================================

def get_ppo_diagnostics(
    ratio: torch.Tensor,
    clip_param: float,
):
    with torch.no_grad():
        ratio_drift = (
            ratio - 1.0
        ).abs().mean()

        clip_fraction = (
            (ratio < 1.0 - clip_param)
            |
            (ratio > 1.0 + clip_param)
        ).float().mean()

    return (
        ratio_drift.item(),
        clip_fraction.item(),
    )


# Example
example_ratio = torch.tensor([
    1.00,
    1.05,
    1.30,
    0.70,
])

drift, clip_fraction = get_ppo_diagnostics(
    example_ratio,
    clip_param=0.2,
)

print("ratio drift:", drift)
print("clip fraction:", clip_fraction)

ratio drift: 0.16249997913837433
clip fraction: 0.5
